# CARR benchmark on Google Colab

Runs the CARR baselines with a GPU runtime (*Runtime → Change runtime type → GPU*).
All caches, checkpoints and results are written to Google Drive, so the notebook can simply be started again
(*Runtime → Run all*) after a disconnection: finished runs are skipped and interrupted runs resume.

1. Put the dataset in Google Drive, either as downloaded from Harvard Dataverse (folder with `carr_manifest.csv`)
   or in the packed layout described in the README (recommended on Drive: zipped images).
2. Set `DATA`, `WORK` and `FAMILY` below and run all cells.

Several Colab sessions (e.g. from different accounts sharing the same Drive folder) can work on the same
family in parallel; set `REVERSE = True` in the second session so it starts from the last run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA = '/content/drive/MyDrive/CARR'          # dataset root (release or packed layout)
WORK = '/content/drive/MyDrive/CARR_work'     # caches, runs and results (kept between sessions)
FAMILY = 'skeleton'                           # 'detection', 'skeleton', 'video', 'reid' or 'all'
MODELS = []                                   # e.g. ['CTR-GCN']; empty = all models of the family
SPLITS = []                                   # e.g. ['CS1', 'CG']; empty = all splits
REVERSE = False

In [ ]:
!git clone -q https://github.com/januaradip/CARR.git /content/CARR 2>/dev/null || git -C /content/CARR pull -q
!pip install -q -r /content/CARR/requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, sys
os.chdir('/content/CARR')
args = ['--data', DATA, '--work', WORK, '--family', FAMILY]
if MODELS: args += ['--models', *MODELS]
if SPLITS: args += ['--splits', *SPLITS]
if REVERSE: args += ['--reverse']
!python run_benchmark.py {' '.join(f'"{a}"' for a in args)} --list

In [ ]:
!python run_benchmark.py {' '.join(f'"{a}"' for a in args)}

## Tables and figures
When all runs are finished, the tables (mean ± standard deviation over folds) and Figs. 8–10 are written to `WORK/figures`.

In [ ]:
!python tools/results_figures.py --work "{WORK}" --out "{WORK}/figures"
import pandas as pd
pd.read_csv(f'{WORK}/figures/results_tables.csv')